# 01. Project: RAG over LLM Notes

**Goal:** Build an end-to-end RAG system over a personal collection of Machine Learning notes.

This project notebook ties together:
- Chunking & embeddings ([01_chunking_embeddings_vectorstore](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems/01_chunking_embeddings_vectorstore.ipynb))
- Basic RAG pipeline ([02_basic_rag_pipeline](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems/02_basic_rag_pipeline.ipynb))
- Optional hybrid ideas from advanced RAG

We will:
1. Load / define a set of **LLM study notes**
2. Chunk, embed, and index them (FAISS)
3. Retrieve relevant passages for a question
4. Generate grounded answers (GPU-first, CPU fallback)
5. Package a simple `ask()` function you can reuse

## 1. Setup

```bash
pip install transformers sentence-transformers faiss-cpu langchain-text-splitters accelerate
```

In [14]:
# pip install transformers sentence-transformers faiss-cpu langchain-text-splitters accelerate

In [15]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


## 2. Your LLM Notes Corpus

In a real project these would be markdown files, PDFs, or Notion exports.  
Here we embed a compact set of notes covering topics from this learning path.

In [16]:
NOTES = [
    {
        "id": "attn",
        "title": "Attention & Transformers",
        "text": """
Scaled dot-product attention scores queries against keys, divides by sqrt(d_k),
applies softmax, and mixes values. Multi-head attention runs several heads in
parallel so different subspaces can specialize. Causal (look-ahead) masks block
future tokens for autoregressive LMs. A Transformer block stacks attention and a
position-wise feed-forward network with residual connections and layer norm.
Pre-LN is common in GPT-style models. Positional encodings (sinusoidal or learned)
inject order because attention alone is permutation-invariant.
""".strip(),
    },
    {
        "id": "ft",
        "title": "Fine-Tuning, LoRA, QLoRA",
        "text": """
Full fine-tuning updates every weight and is memory-heavy. LoRA freezes the base
model and trains low-rank adapters (matrices A and B) with rank r and scaling alpha.
Only a small fraction of parameters are trainable. QLoRA loads the base model in
4-bit NF4 quantization and still trains LoRA adapters in higher precision, enabling
7B–13B instruction tuning on a single consumer GPU. Use PEFT for LoRA config and
SFTTrainer or Trainer for the optimization loop. Always keep train/eval splits and
monitor loss and generation quality.
""".strip(),
    },
    {
        "id": "emb",
        "title": "Embeddings & Similarity",
        "text": """
Sentence embeddings map text to dense vectors so similar meanings are close in
cosine space. all-MiniLM-L6-v2 is a strong, fast default (384-d). Mean pooling over
token states (masked) is a standard way to get one vector per passage. Normalize
embeddings if you use inner-product search as cosine. Good embeddings are the
backbone of dense retrieval in RAG systems.
""".strip(),
    },
    {
        "id": "rag",
        "title": "RAG Systems",
        "text": """
Retrieval-Augmented Generation chunks documents, embeds them, and stores vectors
in FAISS, Chroma, or similar. At query time we embed the question, retrieve top-k
chunks, and pass them as context to a generator with a strict grounded prompt.
Hybrid search mixes dense vectors with BM25 sparse scores (e.g. RRF fusion).
Cross-encoder rerankers improve precision on the top candidates. Always evaluate
retrieval hit rate and whether answers stay faithful to context.
""".strip(),
    },
    {
        "id": "agents",
        "title": "Agents & Tool Use",
        "text": """
ReAct agents interleave reasoning traces with tool actions and observations.
Function calling uses JSON schemas (name, description, parameters) so the model
returns structured calls your code executes. Small models often need constrained
prompts or multiple-choice routing instead of free-form Action lines. Buffer memory
stores recent turns so multi-step dialogues can refer to earlier tool results.
""".strip(),
    },
    {
        "id": "train",
        "title": "Training Practicalities",
        "text": """
Use TrainingArguments for epochs, batch size, learning rate, fp16/bf16, and
evaluation strategy. DataCollatorWithPadding pads dynamically per batch. For causal
LM instruction tuning, format examples as Instruction/Response and mask loss on
prompt tokens when possible. Gradient checkpointing and accumulation reduce memory.
Always set a pad token for GPT-2-style models (often pad=eos).
""".strip(),
    },
]

print(f"{len(NOTES)} notes")
for n in NOTES:
    print(f"  - {n['title']} ({len(n['text'])} chars)")

6 notes
  - Attention & Transformers (541 chars)
  - Fine-Tuning, LoRA, QLoRA (531 chars)
  - Embeddings & Similarity (365 chars)
  - RAG Systems (464 chars)
  - Agents & Tool Use (400 chars)
  - Training Practicalities (386 chars)


## 3. Chunk → Embed → FAISS Index

In [17]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for note in NOTES:
    for i, part in enumerate(splitter.split_text(note["text"])):
        chunks.append({
            "chunk_id": f"{note['id']}_{i}",
            "note_id": note["id"],
            "title": note["title"],
            "text": part.strip(),
        })

print(f"Chunks: {len(chunks)}")

EMBED_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_NAME, device=DEVICE)

texts = [c["text"] for c in chunks]
emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
emb = np.asarray(emb, dtype="float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print(f"FAISS vectors: {index.ntotal}, dim={emb.shape[1]}")

Chunks: 18


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS vectors: 18, dim=384


## 4. Retriever

In [18]:
def retrieve(query: str, k: int = 4):
    q = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.asarray(q, dtype="float32"), k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[int(idx)]
        hits.append({
            "score": float(score),
            "title": c["title"],
            "text": c["text"],
            "chunk_id": c["chunk_id"],
            "note_id": c["note_id"],
        })
    return hits


for h in retrieve("What is LoRA?", k=3):
    print(f"[{h['score']:.3f}] {h['title']}: {h['text'][:90]}...")

[0.535] Fine-Tuning, LoRA, QLoRA: Full fine-tuning updates every weight and is memory-heavy. LoRA freezes the base
model and...
[0.255] Fine-Tuning, LoRA, QLoRA: 7B–13B instruction tuning on a single consumer GPU. Use PEFT for LoRA config and
SFTTraine...
[0.228] Fine-Tuning, LoRA, QLoRA: Only a small fraction of parameters are trainable. QLoRA loads the base model in
4-bit NF4...


## 5. Generator (v5-compatible Seq2Seq)

Uses `AutoModelForSeq2SeqLM.generate()` — not the removed `text2text-generation` pipeline.

In [19]:
GEN_NAME = "google/flan-t5-large" if DEVICE == "cuda" else "google/flan-t5-small" # switch to "google/flan-t5-base" if results are not good
gen_tok = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_NAME).to(DEVICE)
gen_model.eval()
print(f"Generator: {GEN_NAME}")


def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs = gen_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gen_tok.decode(out[0], skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generator: google/flan-t5-small


## 6. Grounded RAG Prompt + `ask()`

In [20]:
def build_prompt(question: str, hits: list) -> str:
    ctx = "\n\n".join(
        f"[{i}] ({h['title']}) {h['text']}" for i, h in enumerate(hits, 1)
    )
    return (
        "You are a study assistant. Answer ONLY using the notes below. "
        "If the notes do not contain the answer, say "
        "\"I don't know based on the provided notes.\"\n\n"
        f"Notes:\n{ctx}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def ask(question: str, k: int = 4, show_sources: bool = True) -> str:
    hits = retrieve(question, k=k)
    if show_sources:
        print("Sources:")
        for h in hits:
            print(f"  [{h['score']:.3f}] {h['title']} ({h['chunk_id']})")
        print()
    prompt = build_prompt(question, hits)
    answer = generate(prompt)
    return answer


print("ask() ready.")

ask() ready.


## 7. Try Study Questions

In [23]:
questions = [
    "What is multi-head attention and why use it?",
    "Explain LoRA and QLoRA in simple terms.",
    "How does a RAG system work at query time?",
    "What is Pre-LN in Transformer blocks?",
    "How do agents use tools and memory?",
    "Who won the 2012 Wimbledon?",
    "Who won the 2010 World Cup?",  # should refuse / not in notes
]

for q in questions:
    print("=" * 60)
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")

Q: What is multi-head attention and why use it?
Sources:
  [0.600] Attention & Transformers (attn_0)
  [0.417] Attention & Transformers (attn_3)
  [0.300] Attention & Transformers (attn_1)
  [0.241] Agents & Tool Use (agents_2)

A: run several heads in [2] (Attention & Transformers) inject order because attention alone is permutation-invariant.

Q: Explain LoRA and QLoRA in simple terms.
Sources:
  [0.441] Fine-Tuning, LoRA, QLoRA (ft_1)
  [0.401] Fine-Tuning, LoRA, QLoRA (ft_0)
  [0.158] Fine-Tuning, LoRA, QLoRA (ft_2)
  [0.153] Embeddings & Similarity (emb_0)

A: [3]

Q: How does a RAG system work at query time?
Sources:
  [0.413] Embeddings & Similarity (emb_1)
  [0.311] RAG Systems (rag_0)
  [0.307] Agents & Tool Use (agents_2)
  [0.248] Attention & Transformers (attn_1)

A: We embed the question, retrieve top-k [3] (Agents & Tool Use) stores recent turns so multi-step dialogues can refer to earlier tool results. [4] (Attention & Transformers) parallel so different subspaces can sp

## 8. Optional: Save / Reload the Index

For a personal notes app you would persist chunks + FAISS to disk.

In [22]:
import pickle
from pathlib import Path

save_dir = Path("./llm_notes_rag")
save_dir.mkdir(exist_ok=True)

faiss.write_index(index, str(save_dir / "notes.faiss"))
with open(save_dir / "chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Saved to", save_dir.resolve())

# Reload sketch:
# index = faiss.read_index("llm_notes_rag/notes.faiss")
# chunks = pickle.load(open("llm_notes_rag/chunks.pkl", "rb"))

Saved to /content/llm_notes_rag


## 9. How to Point This at *Your* Notes

1. Put markdown / text files in a folder (e.g. `./my_notes/`).
2. Load them:

```python
from pathlib import Path
NOTES = []
for path in Path("./my_notes").glob("**/*.md"):
    NOTES.append({
        "id": path.stem,
        "title": path.stem.replace("_", " ").title(),
        "text": path.read_text(encoding="utf-8"),
    })
```

3. Re-run chunk → embed → index → `ask()`.

Tips:
- Prefer recursive chunking with modest overlap.
- Keep titles / filenames in metadata for citations.
- For large corpora, use Chroma/Qdrant and hybrid search + reranking.

## 10. Summary

| Step | Implementation |
|------|----------------|
| Notes corpus | List of `{id, title, text}` (or files on disk) |
| Chunking | `RecursiveCharacterTextSplitter` |
| Embeddings | `all-MiniLM-L6-v2` |
| Index | FAISS `IndexFlatIP` |
| Retrieve | Top-k cosine / IP |
| Generate | FLAN-T5 via `generate()` |
| API | `ask(question)` with source display |

You now have a **personal LLM-notes tutor** pattern you can grow into a full study assistant.

---

**Alternative project MPhil ML notes notebook** [`01_rag_over_ml_notes.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/06_projects/01_rag_over_ml_notes.ipynb)

**Next project notebook:** [`02_hybrid_classical_plus_llm.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/06_projects/02_hybrid_classical_plus_llm.ipynb)
Combining Classical ML (PCA, GMM, etc.) with LLMs

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Science/Analytics and ML/AI related opportunities

---